# 读取数据

In [ ]:
# 1. 数据准备：读取类别 + 动态生成 8000 张抽样清单
import json
import random
import yaml
from pathlib import Path

# 原始挂载路径
DATA_ROOT = Path("/kaggle/input/datasets/shilljinglza/traffic-data/TrafficDetYolo")
DATA_YAML = str(DATA_ROOT / "data.yaml")

# 读取类别并保存
names = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))["names"]
json.dump(names, open("/kaggle/working/traffic_classes.json", "w"), ensure_ascii=False, indent=2)

print("✅ 数据就绪:", DATA_ROOT)
print("类别列表:", names)

# 从 train 图片中随机抽取 8000 张
train_img_dir = DATA_ROOT / "train" / "images"
all_imgs = [f.name for f in train_img_dir.iterdir() if f.suffix.lower() in ['.jpg', '.jpeg', '.png']]

random.seed(42)  # 固定种子，保证结果可复现
sample_imgs = random.sample(all_imgs, min(8000, len(all_imgs)))

# 把抽样出来的图片路径写入一个 txt 清单
sample_txt_path = "/kaggle/working/train_sample.txt"
with open(sample_txt_path, "w") as f:
    for img in sample_imgs:
        f.write(f"{DATA_ROOT}/train/images/{img}\n")

data = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
data["path"] = str(DATA_ROOT)
data["train"] = sample_txt_path  # 指向刚才生成的清单
data["val"] = "val/images"

SAMPLE_YAML = "/kaggle/working/traffic_sample_v8.yaml"
with open(SAMPLE_YAML, "w") as f:
    yaml.dump(data, f, sort_keys=False)

print(f"\n✅ 抽样配置生成完毕！")
print(f"本次训练图片数: {len(sample_imgs)} 张")
print(f"后续训练请使用该配置: {SAMPLE_YAML}")

In [ ]:
# 2. 安装 ultralytics + 预下载 yolov8s.pt
import importlib.util
import subprocess
import sys
from pathlib import Path

# Kaggle 通常自带 ultralytics，如果没有则安装
if importlib.util.find_spec("ultralytics") is None:
    print("正在安装 ultralytics...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics"], check=True)

# 下载预训练权重（如果需要）
if not Path("/kaggle/working/yolov8s.pt").exists():
    print("正在下载 yolov8s.pt...")
    subprocess.run(["wget", "-q", "-P", "/kaggle/working",
                    "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8s.pt"], check=True)

print("✅ 环境准备完毕")

In [ ]:
# 3. 实验配置与训练（导包直接调用）
import torch
from ultralytics import YOLO

# 超参数设置
EPOCHS, IMGSZ, BATCH, SEED = 5, 320, 4, 42
PROJECT_DIR = "/kaggle/working/traffic_models_v8"

RUNS = [
    dict(name="v8_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),  # 基线
    dict(name="v8_lr0.005",                 lr=0.005, wd=0.0005),  # 学习率消融
    dict(name="v8_lr0.001",                 lr=0.001, wd=0.0005),  # 学习率消融
]
BEST_LR = 0.01
for _wd in (0.001, 0.0001):
    RUNS.append(dict(name=f"v8_wd{_wd}", lr=BEST_LR, wd=_wd))

for run in RUNS:
    print(f"\n{'='*60}")
    print(f"🚀 开始训练: {run['name']} (lr={run['lr']}, wd={run['wd']})")
    print(f"{'='*60}\n")
    
    model = YOLO("/kaggle/working/yolov8s.pt") 
    
    model.train(
        data="/kaggle/working/traffic_sample_v8.yaml", 
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        seed=SEED,
        optimizer='SGD',
        lr0=run['lr'],           # YOLOv8 原生支持直接传参
        lrf=1.0,                 # 关闭余弦衰减，保证公平对比
        weight_decay=run['wd'],  # 原生支持
        project=PROJECT_DIR,
        name=run['name'],
        exist_ok=True,
        device=0,                
        workers=2,               
        cache=False,             
        verbose=False
    )

print("\n✅ 全部 YOLOv8 消融实验训练完成！")

In [ ]:
# 4. 汇总各次实验指标
import pandas as pd
import json
from pathlib import Path

# 自动寻找正确的实验输出目录
RUN_DIR = Path("/kaggle/working/runs_v8")
if not RUN_DIR.exists():
    RUN_DIR = Path("/kaggle/working/traffic_models_v8") 

RUNS = [
    dict(name="v8_baseline_lr0.01_wd0.0005", lr=0.01,  wd=0.0005),
    dict(name="v8_lr0.005",                 lr=0.005, wd=0.0005),
    dict(name="v8_lr0.001",                 lr=0.001, wd=0.0005),
    dict(name="v8_wd0.001",                 lr=0.01,  wd=0.001),
    dict(name="v8_wd0.0001",                lr=0.01,  wd=0.0001),
]

def pick_col(df, *needles):
    for c in df.columns:
        if all(k in c for k in needles):
            return c
    raise KeyError(f"未找到包含 {needles} 的列，请检查 results.csv")

def parse_run(name, lr, wd):
    # 兼容多种路径形式
    possible_paths = [
        RUN_DIR / name / "results.csv",
        Path("/kaggle/working") / name / "results.csv"
    ]
    csv_path = None
    for p in possible_paths:
        if p.exists():
            csv_path = p
            break
            
    if not csv_path:
        print(f"⚠️ 跳过 {name}: 未找到结果文件 (可能是没跑完或被中断)")
        return None
        
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    
    # 容错：如果缺少 epoch 列，自动补齐
    if "epoch" not in df.columns:
        df.insert(0, "epoch", range(len(df)))
        
    try:
        c_ep = pick_col(df, "epoch")
        c_p = pick_col(df, "precision")
        c_r = pick_col(df, "recall")
        c_m50 = pick_col(df, "mAP50")
        c_m5095 = pick_col(df, "mAP50-95")
    except KeyError as e:
        print(f"⚠️ 解析 {name} 失败: {e}")
        return None

    best_i = df[c_m50].idxmax()
    best = df.loc[best_i]
    
    return dict(name=name, lr=lr, wd=wd, epochs=int(best[c_ep]) + 1,
                best_epoch=int(best[c_ep]) + 1,
                precision=float(best[c_p]),
                recall=float(best[c_r]),
                mAP50=float(best[c_m50]),
                mAP5095=float(best[c_m5095]),
                curves=dict(epoch=df[c_ep].astype(int).tolist(),
                            mAP50=df[c_m50].tolist(),
                            mAP5095=df[c_m5095].tolist()))

# 过滤掉未跑完的实验
summaries = [s for s in (parse_run(**r) for r in RUNS) if s is not None]

if summaries:
    tbl = pd.DataFrame(summaries)[["name", "lr", "wd", "precision", "recall", "mAP50", "mAP5095"]]
    print("📊 消融实验结果汇总（按 mAP50 降序排列）")
    print("="*60)
    print(tbl.sort_values("mAP50", ascending=False).to_string(index=False))
    
    json.dump(summaries, open("/kaggle/working/yolov8_runs_summary.json", "w"), ensure_ascii=False, indent=2)
    
    best = max(summaries, key=lambda s: s["mAP50"])
    print(f"\n🏆 表现最好的参数组合: {best['name']}")
    print(f"👉 最优 lr = {best['lr']}, 最优 wd = {best['wd']}, 最高 mAP50 = {best['mAP50']:.4f}")
else:
    print("❌ 没有找到任何有效实验结果")

In [ ]:
import shutil, pathlib
best = max(summaries, key=lambda s: s["mAP50"])
src = pathlib.Path("/kaggle/working/traffic_models_v8") / best["name"] / "weights/best.pt"
shutil.copy2(src, "/kaggle/working/yolov8s_traffic_best.pt")
print(f"导出完成: {best['name']} (mAP50={best['mAP50']:.4f})")

In [ ]:
# 消融对比
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
for s in summaries:
    plt.plot(s["curves"]["epoch"], s["curves"]["mAP50"], marker="o", label=s["name"])
plt.xlabel("epoch"); plt.ylabel("mAP@0.5"); plt.grid(alpha=.3); plt.legend(fontsize=8)
plt.title("YOLOv8s ablation: mAP@0.5")
plt.tight_layout()
plt.savefig("/kaggle/working/yolov8_ablation_curves.png", dpi=150)
plt.show()

In [ ]:
import yaml
from pathlib import Path

# 你的原始数据集挂载路径
DATA_ROOT = "/kaggle/input/datasets/shilljinglza/traffic-data/TrafficDetYolo"
SRC_YAML = Path(DATA_ROOT) / "data.yaml"

# 读取原 yaml
with open(SRC_YAML, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

# 强制改成绝对路径
data["path"] = DATA_ROOT
data["train"] = "train/images"
data["val"] = "val/images"

# 写入到可写目录
NEW_YAML = "/kaggle/working/traffic_data_abs_v8.yaml"
with open(NEW_YAML, "w", encoding="utf-8") as f:
    yaml.dump(data, f, sort_keys=False)

print(f"✅ 绝对路径 yaml 已重新生成: {NEW_YAML}")
print(open(NEW_YAML).read())

In [ ]:
from ultralytics import YOLO

# 消融实验得出的最优参数
BEST_LR = 0.001
BEST_WD = 0.0005

# 加载预训练权重
model = YOLO("/kaggle/working/yolov8s.pt")

model.train(
    # 全量数据的绝对路径 yaml
    data="/kaggle/working/traffic_data_abs_v8.yaml",
    
    # 分辨率提升到 640
    imgsz=640,
    
    epochs=20,
    patience=10,
    
    optimizer='SGD',
    lr0=BEST_LR,           # 最优学习率
    lrf=0.1,               # 余弦退火
    weight_decay=BEST_WD,  # 最优权重衰减
    
    # 防止 OOM 显存爆炸，batch 降为 8
    batch=8,
    seed=42,
    
    # 结果保存路径
    project="/kaggle/working/traffic_models_v8_full",
    name="yolov8s_full_best",
    exist_ok=True,
    
    device=0,              # 使用 GPU
    workers=2,             # 数据加载线程
    cache=False            # 保持关闭，规避只读目录报错
)

In [ ]:
import shutil
import os
from pathlib import Path

# 你的训练结果保存目录
RESULT_DIR = Path("/kaggle/working/traffic_models_v8_full/yolov8s_full_best")
# 你想生成的压缩包路径
ZIP_PATH = "/kaggle/working/yolov8s_full_best_results.zip"

if not RESULT_DIR.exists():
    print(f"❌ 找不到训练结果目录: {RESULT_DIR}")
    print("请确认训练是否已经跑完，或者检查 project 和 name 参数是否写对。")
else:
    # 删除旧的压缩包（如果存在）
    if os.path.exists(ZIP_PATH):
        os.remove(ZIP_PATH)
    
    # 打包整个结果目录
    print(f"正在打包: {RESULT_DIR} ...")
    shutil.make_archive(
        base_name=ZIP_PATH.replace(".zip", ""),  # 去掉后缀，make_archive 会自动加 .zip
        format='zip',
        root_dir=RESULT_DIR
    )
    
    # 打印结果
    zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
    print(f"\n✅ 打包完成！")
    print(f"压缩包路径: {ZIP_PATH}")
    print(f"压缩包大小: {zip_size_mb:.1f} MB")
    print("\n📦 压缩包内包含：")
    print("  - weights/best.pt  (最佳模型权重)")
    print("  - weights/last.pt  (最后一个epoch的权重)")
    print("  - results.csv      (训练过程数据，画 Loss/mAP 曲线用)")
    print("  - confusion_matrix.png  (混淆矩阵)")
    print("  - F1_curve.png, PR_curve.png 等评估图表")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RUN_DIR = Path("/kaggle/working/traffic_models_v8")

RUNS = [
    "v8_baseline_lr0.01_wd0.0005",
    "v8_lr0.005",
    "v8_lr0.001",
    "v8_wd0.001",
    "v8_wd0.0001",
]

plt.figure(figsize=(10, 6))
for name in RUNS:
    csv_path = RUN_DIR / name / "results.csv"
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df.columns = [c.strip() for c in df.columns]
        plt.plot(df["epoch"], df["metrics/mAP50(B)"], marker='o', label=name)

plt.xlabel("Epoch")
plt.ylabel("mAP@0.5")
plt.title("YOLOv8s Ablation Study: mAP@0.5 Comparison")
plt.grid(alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("/kaggle/working/ablation_comparison.png", dpi=200)
plt.show()